In [5]:
pip install torch numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
import csv
import random

rows = 847
file_path = "replay_buffer.csv"

def image_vector():
    return " ".join(f"{random.uniform(0,1):.4f}" for _ in range(4096))

with open(file_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["state","steering","throttle","reward","next_state","done"])
    for _ in range(rows):
        writer.writerow([
            image_vector(),
            round(random.uniform(-1,1),2),
            round(random.uniform(0,1),2),
            round(random.uniform(-10,2),2),
            image_vector(),
            random.choice([True, False])
        ])


In [2]:
import csv
import numpy as np

states = []
actions = []
rewards = []
next_states = []
dones = []

with open("replay_buffer.csv", "r", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)  # skip header

    for row in reader:
        state_str, steering, throttle, reward, next_state_str, done = row

        state = np.array(
            state_str.split(), dtype=np.float32
        )
        state = np.array(
            state_str.split(), dtype=np.float32
        )


        states.append(state)
        actions.append([float(steering), float(throttle)])
        rewards.append(float(reward))
        dones.append(done == "True")

states = np.array(states)
actions = np.array(actions, dtype=np.float32)
rewards = np.array(rewards, dtype=np.float32)
dones = np.array(dones, dtype=np.bool_)

print(states.shape)  # (847, 4096)


(847, 4096)


In [3]:
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
states = []
actions = []
rewards = []

with open("replay_buffer.csv", "r") as f:
    reader = csv.reader(f)
    header = next(reader)

    for row in reader:
        state_str, steering, throttle, reward, _, _ = row

        state = np.array(state_str.split(), dtype=np.float32)
        action = np.array([float(steering), float(throttle)], dtype=np.float32)

        states.append(state)
        actions.append(action)
        rewards.append(float(reward))

states = torch.tensor(np.array(states))
actions = torch.tensor(np.array(actions))
rewards = torch.tensor(np.array(rewards))


In [5]:
class PolicyANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4096, 1024),
            nn.ReLU(),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, 2)  # steering, throttle
        )

    def forward(self, x):
        return self.net(x)
policy = PolicyANN()
optimizer = optim.Adam(policy.parameters(), lr=1e-4)

In [6]:
test_output = policy(states[0])
print(test_output)


tensor([-0.0252, -0.0942], grad_fn=<ViewBackward0>)


In [7]:
for epoch in range(10):
    total_loss = 0.0

    for i in range(len(states)):
        state = states[i]
        action = actions[i]
        reward = rewards[i]

        predicted_action = policy(state)

        # Encourage actions that gave high reward
        loss = -reward * torch.mean((predicted_action - action) ** 2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")


Epoch 1 | Loss: 820.9687
Epoch 2 | Loss: 764.2416
Epoch 3 | Loss: 742.5505
Epoch 4 | Loss: 710.5325
Epoch 5 | Loss: 664.0324
Epoch 6 | Loss: 582.0707
Epoch 7 | Loss: 473.3377
Epoch 8 | Loss: 273.5340
Epoch 9 | Loss: 163.0458
Epoch 10 | Loss: 120.1524
